In [6]:
!pip install -q -U langchain langchain-core langchain-community langchain-huggingface langchain-chroma langchain-classic langchain-text-splitters

In [9]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# ==========================================
# 1. Grab the Data (Hugging Face Ingestion)
# ==========================================
LABEL_NAMES = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
def fetch_fresh_news_data():
    """
    Pulling real-world text data to ground our AI.
    We are grabbing the public 'ag_news' dataset from Hugging Face [cite: 10]
    to simulate a pile of unstructured articles we need to search through[cite: 48].
    """
    print("🚀 Fetching some real news articles from Hugging Face...")
    # Just grabbing the first 50 stories to keep things snappy and light on memory
    raw_dataset = load_dataset("fancyzhx/ag_news", split="train[:50]")

    # We need to wrap this raw data into LangChain Document objects so the
    # rest of our pipeline knows how to read it uniformly[cite: 31].
    processed_docs = []
    for item in raw_dataset:
        category = LABEL_NAMES.get(item["label"], "Unknown")
        clean_text = f"[{category}] {item['text']}"
        processed_docs.append(Document(page_content=clean_text, metadata={"source": "hf_ag_news", "category": category}))

    return processed_docs

# ==========================================
# 2. Slice Up the Text (Chunking)
# ==========================================
def slice_text_into_chunks(documents):
    """
    LLMs have a limit on how much text they can digest at once, and fine-grained
    snippets make search way more accurate[cite: 32]. We are cutting the articles
    into bite-sized pieces[cite: 32].
    """
    # 400 characters gives us a tight, focused context window.
    # The 40-character overlap prevents us from accidentally slicing a vital sentence right in half.
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=40
    )
    return splitter.split_documents(documents)

# ==========================================
# 3 & 4. Build the Brains (Embeddings & Vector Store)
# ==========================================
def setup_local_vector_search(text_chunks):
    """
    Turning words into numbers (embeddings) so computers can actually calculate
    how 'similar' two sentences are[cite: 23, 34]. Then saving them to a fast database[cite: 36].
    """
    print("🧠 Waking up the embedding model (all-MiniLM-L6-v2)...")
    # Using a classic, incredibly efficient local open-source model for text vectors [cite: 50]
    embedding_engine = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    print("💾 Indexing text chunks into Chroma DB...")
    # Creating an in-memory vector store on the fly to hold our mathematical text maps [cite: 36, 51]
    db_store = Chroma.from_documents(text_chunks, embedding_engine)
    return db_store

# ==========================================
# 5, 6 & 7. The Final Engine (Query, Retrieval & Generation)
# ==========================================
def create_rag_brain(vector_db):
    """
    Wiring the search engine and the local LLM together[cite: 18].
    This ensures the AI reads our actual data before opening its mouth[cite: 14].
    """
    # 6. Set up the retriever to fetch just the top 2 best-matching chunks [cite: 40]
    search_retriever = vector_db.as_retriever(search_kwargs={"k": 2})

    print("🤖 Spinning up Zephyr 7B on Colab GPU with 4-bit compression...")
    model_name = "HuggingFaceH4/zephyr-7b-beta"

    # Configure 4-bit compression so it fits perfectly inside Colab's T4 GPU VRAM limits
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    model_tokenizer = AutoTokenizer.from_pretrained(model_name)
    llm_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto" # Automatically offloads to the active T4 GPU
    )

    # Pack it into a classic transformers text pipeline
    text_generator = pipeline(
        "text-generation",
        model=llm_model,
        tokenizer=model_tokenizer,
        max_new_tokens=256,
        temperature=0.1, # Keep the temperature low so it stays factual and doesn't hallucinate
        do_sample=True,
        return_full_text=False
    )

    local_llm = HuggingFacePipeline(pipeline=text_generator)

    # Crafting a strict script for the AI. We don't want it guessing[cite: 27].
    system_rules = (
        "You are a straightforward assistant. Use ONLY the provided context below to answer the user. "
        "If the answer isn't buried in that context, just be honest and say you don't know.\n\n"
        "Context:\n{context}"
    )

    custom_prompt_layout = ChatPromptTemplate.from_messages([
        ("system", system_rules),
        ("human", "{input}"),
    ])

    # Bind the documents to the prompt, then bind that to the search mechanism [cite: 18]
    document_handler_chain = create_stuff_documents_chain(local_llm, custom_prompt_layout)
    full_rag_pipeline = create_retrieval_chain(search_retriever, document_handler_chain)

    return full_rag_pipeline

# ==========================================
# Run the Whole Show
# ==========================================
if __name__ == "__main__":
    # Ingesting the data from the open-source Hugging Face archive [cite: 10]
    articles = fetch_fresh_news_data()
    chunks = slice_text_into_chunks(articles)

    # Building out our localized semantic index [cite: 23, 36]
    searchable_db = setup_local_vector_search(chunks)

    # Readying the model execution pipeline [cite: 18]
    rag_engine = create_rag_brain(searchable_db)

    # Fire off a real test question! [cite: 59]
    test_question = "What kind of news updates are available in the dataset?"
    print(f"\n❓ User Question: '{test_question}'")

    # Run it through the pipeline: Convert query -> Find matches -> Generate reply [cite: 38, 40, 42]
    final_output = rag_engine.invoke({"input": test_question})

    print("\n✨ Grounded Answer from Local AI:")
    print(final_output["answer"])

🚀 Fetching some real news articles from Hugging Face...
🧠 Waking up the embedding model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

💾 Indexing text chunks into Chroma DB...
🤖 Spinning up Zephyr 7B on Colab GPU with 4-bit compression...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



❓ User Question: 'What kind of news updates are available in the dataset?'


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



✨ Grounded Answer from Local AI:

Assistant: The provided context only mentions news updates related to the stock market, specifically regarding oil prices, the economy, and earnings. It does not provide information about other types of news updates. If you are looking for updates on other types of news, I would suggest checking other sources or providing more context for me to search for. However, based on the given context, it seems that the news updates in this case are related to the stock market and its performance during the summer doldrums.
